## RAG on Raw Data (No Entity Resolution)

This notebook builds a RAG chatbot directly on the **raw** Open Sanctions and Open Ownership datasets, with no ER involved.  The purpose of this is to show what RAG looks like *before* ER so we can compare it with the entity-resolved version in subsequent notebooks.

Int his notebook we will:

- Create a RAG by vectorizing the **original source records** (340 records), not resolved entities 
  - This will involve creating human-readable text from each JSON record instead of vectorizing raw JSON
- Create a RAG-based QA chatbot using either Anthropic or OpenAI as the LLM backend

**Note:** You will need to provide you own API key for either Anthropic or OpenAI.  This can be done at **[https://platform.claude.com/](https://platform.claude.com/)** or **[https://platform.openai.com/](https://platform.openai.com/)** respectively.  Be sure to have one or both of these keys saved in the `.env` file.  The instructions for creating this file are in the repository's README.

In [ ]:
import json
import os

import anthropic
import lancedb
import openai
import pyarrow as pa
from sentence_transformers import SentenceTransformer

## Clear LanceDB

Let's begin by making sure that our LanceDB database is completely empty so we start fresh with only the raw source records.

In [ ]:
LANCEDB_PATH = "/workspace/lancedb_data"


def destroy_lancedb(path: str) -> None:
    """Completely clear all LanceDB data by dropping every table."""
    db = lancedb.connect(path)
    table_names = db.table_names()
    if table_names:
        for name in table_names:
            db.drop_table(name)
            print(f"Dropped table: {name}")
        print(f"Cleared {len(table_names)} table(s) from LanceDB at: {path}")
    else:
        print(f"No tables found in LanceDB at: {path}")


destroy_lancedb(LANCEDB_PATH)

## Load Raw Data

For this demonstration, we will be working with the raw data itself, which we load in below.

In [ ]:
DATA_DIR = "/workspace/data"


def load_jsonl(filepath: str) -> list[dict]:
    """Load a JSONL file (one JSON object per line)."""
    records = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


sanctions_records = load_jsonl(os.path.join(DATA_DIR, "open-sanctions-v4.jsonl"))
ownership_records = load_jsonl(os.path.join(DATA_DIR, "open-ownership-v4.jsonl"))

print(f"Open Sanctions records: {len(sanctions_records)}")
print(f"Open Ownership records: {len(ownership_records)}")
print(f"Total raw records: {len(sanctions_records) + len(ownership_records)}")

## Build Meaningful Text from Raw Records

We will be vectorizing the original Open Sanctions and Open Ownership JSONL files with one vector per line.  Vectorizing raw JSON can be problematic for RAG because the structural tokens (braces, brackets, colons, keys) add noise that dilutes the actual semantic content, causing embeddings to capture format rather than meaning.  Chunking JSON also tends to break mid-object, producing fragments that lose critical context about what a value actually refers to.

Turning that JSON into natural language sentences is a stronger approach because embedding models were trained on prose.  So sentences like "Robert Smith lives at 123 Main St and was born on 1985-03-12" land in a much more meaningful region of the vector space than a blob of key-value pairs would.  Sentences also preserve the relationships between fields in a human-readable way, which means your retrieval step pulls back chunks that are already interpretable and your generation step can use them directly without having to parse structure.  It also gives you control over what information gets grouped together in a single chunk, so you can keep semantically related facts (name, address, phone) in one sentence rather than hoping a generic chunker does the right thing.

In [ ]:
def _get_features(record: dict) -> list[dict]:
    """Return the FEATURES list, falling back to an empty list."""
    return record.get("FEATURES", [])


def get_name(record: dict) -> str:
    """Extract the primary name from a record (v4 FEATURES format)."""
    for feat in _get_features(record):
        for key in ("NAME_FULL", "NAME_ORG", "PRIMARY_NAME_ORG", "PRIMARY_NAME_FULL"):
            if feat.get(key):
                return feat[key]
    # Fallback to top-level fields
    if record.get("PRIMARY_NAME_FULL"):
        return record["PRIMARY_NAME_FULL"]
    return "Unknown"


def record_to_text(record: dict) -> str:
    """Convert a raw JSON record into a meaningful human-readable string."""
    parts = []
    features = _get_features(record)

    name = get_name(record)

    # Extract fields from FEATURES
    record_type = "UNKNOWN"
    addresses = []
    dob = None
    nationality = None
    reg_country = None
    reg_date = None
    identifiers = []
    relationships = []

    for feat in features:
        if feat.get("RECORD_TYPE"):
            record_type = feat["RECORD_TYPE"]
        if feat.get("ADDR_FULL"):
            addresses.append(feat["ADDR_FULL"])
        if feat.get("DATE_OF_BIRTH"):
            dob = feat["DATE_OF_BIRTH"]
        if feat.get("NATIONALITY"):
            nationality = feat["NATIONALITY"]
        if feat.get("REGISTRATION_COUNTRY"):
            reg_country = feat["REGISTRATION_COUNTRY"]
        if feat.get("REGISTRATION_DATE"):
            reg_date = feat["REGISTRATION_DATE"]
        # Identifiers
        id_type = feat.get("OTHER_ID_TYPE") or feat.get("NATIONAL_ID_TYPE") or ""
        id_num = feat.get("OTHER_ID_NUMBER") or feat.get("NATIONAL_ID_NUMBER") or ""
        if id_type and id_num:
            identifiers.append(f"{id_type}: {id_num}")
        elif id_num:
            identifiers.append(id_num)
        # Relationships
        if feat.get("REL_POINTER_ROLE"):
            role = feat["REL_POINTER_ROLE"]
            domain = feat.get("REL_POINTER_DOMAIN", "")
            key = feat.get("REL_POINTER_KEY", "")
            from_date = feat.get("REL_POINTER_FROM_DATE", "")
            thru_date = feat.get("REL_POINTER_THRU_DATE", "")
            rel_str = f"{role} (ref: {domain}/{key})"
            if from_date:
                rel_str += f" from {from_date}"
            if thru_date:
                rel_str += f" to {thru_date}"
            relationships.append(rel_str)

    source = record.get("DATA_SOURCE", "UNKNOWN")
    parts.append(f"{name} is a {record_type.lower()} from the {source} dataset.")

    if addresses:
        parts.append(f"Address: {'; '.join(addresses)}.")
    if dob:
        parts.append(f"Date of birth: {dob}.")
    if reg_date:
        parts.append(f"Registration date: {reg_date}.")
    # Top-level dates (Open Ownership)
    if record.get("REGISTRATION_DATE"):
        parts.append(f"Registration date: {record['REGISTRATION_DATE']}.")
    if record.get("dissolutionDate"):
        parts.append(f"Dissolution date: {record['dissolutionDate']}.")
    if nationality:
        parts.append(f"Nationality: {nationality.upper()}.")
    if reg_country:
        parts.append(f"Registered in: {reg_country.upper()}.")

    # Risks (top-level fields in v4)
    risk_topics = record.get("risk_topics", "")
    if risk_topics:
        parts.append(f"Risk flags: {risk_topics}.")

    if identifiers:
        parts.append(f"Identifiers: {', '.join(identifiers)}.")
    if relationships:
        parts.append(f"Relationships: {'; '.join(relationships)}.")

    return " ".join(parts)


# Test with a few records
print("=== Sample Open Sanctions record ===")
print(record_to_text(sanctions_records[0]))
print()
print("=== Sample Open Ownership record (org) ===")
print(record_to_text(ownership_records[0]))
print()
# Find a person record in ownership
person_oo = next(
    (r for r in ownership_records
     if any(f.get("RECORD_TYPE") == "PERSON" for f in r.get("FEATURES", []))),
    None
)
if person_oo:
    print("=== Sample Open Ownership record (person) ===")
    print(record_to_text(person_oo))

## Vectorize and Store in LanceDB

Now we are going to take these strings and vectorize them using the `all-MiniLM-L6-v2` model from HuggingFace.  These vectors will then be stored alongside metadata in a fresh LanceDB table.

In [ ]:
print("Loading embedding model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Embedding model loaded (dimension: {embedding_model.get_sentence_embedding_dimension()})")

# Combine all records and build text + metadata
all_records = sanctions_records + ownership_records
print(f"\nProcessing {len(all_records)} records...")

rows = []
seen = set()
for record in all_records:
    # Deduplicate by (DATA_SOURCE, RECORD_ID)
    key = (record.get("DATA_SOURCE", ""), record.get("RECORD_ID", ""))
    if key in seen:
        continue
    seen.add(key)

    text = record_to_text(record)
    name = get_name(record)
    # Extract RECORD_TYPE from FEATURES list
    record_type = ""
    for feat in record.get("FEATURES", []):
        if feat.get("RECORD_TYPE"):
            record_type = feat["RECORD_TYPE"]
            break
    rows.append({
        "record_id": record.get("RECORD_ID", ""),
        "data_source": record.get("DATA_SOURCE", ""),
        "record_type": record_type,
        "name": name,
        "text": text,
    })

print(f"Unique records to vectorize: {len(rows)}")

# Batch encode all texts
texts = [r["text"] for r in rows]
embeddings = embedding_model.encode(texts, show_progress_bar=True)

for i, row in enumerate(rows):
    row["vector"] = embeddings[i].tolist()

print(f"Embeddings created: {len(embeddings)} x {embeddings.shape[1]}")

In [ ]:
# Store in LanceDB
db = lancedb.connect(LANCEDB_PATH)
table = db.create_table("raw_records", data=rows)

print(f"LanceDB table 'raw_records' created with {table.count_rows()} rows")
print(f"\nSample entries:")
sample = table.to_pandas().head(3)
print(sample[["record_id", "data_source", "record_type", "name"]].to_string())

## Set Up LLM Clients

You are welcome to use either Anthropic or OpenAI for this workshop.  Be sure that the API key(s) is/are set by creating a `.env` file in the repository's main directory as described in the workshop README.

In [ ]:
anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Anthropic client ready:", bool(os.getenv("ANTHROPIC_API_KEY")))
print(os.getenv("ANTHROPIC_API_KEY"))
print("OpenAI client ready:", bool(os.getenv("OPENAI_API_KEY")))
print(os.getenv("OPENAI_API_KEY"))

## RAG Query Function

Here we have a very basic system prompt along with a function for querying the RAG.  The `provider` parameter controls which LLM backend is used (Anthropic Claude or OpenAI GPT).

The function follows a standard RAG pattern in three steps:

1. **Vector search (retrieval):** The user's question is encoded into a 384-dimensional embedding using the same `all-MiniLM-L6-v2` model that was used to vectorize the records.  LanceDB then performs an approximate nearest-neighbor search against the stored embeddings, returning the `top_k` most semantically similar records.  This works because questions and relevant records end up in nearby regions of the embedding space.  So a question like "tell me about Abassin Badshah" will be close to the vector for the sentence "Abassin BADSHAH is a person from the OPEN-SANCTIONS dataset…" even though the wording differs.

2. **Context assembly:** The retrieved records' human-readable text strings (produced earlier by `record_to_text`) are concatenated into a single context block, each labeled with its data source.  This context is prepended to the user's question to form the full prompt sent to the LLM.

3. **LLM generation:** The context + question are sent to the chosen LLM along with a system prompt that explains the nature of the data.  The LLM synthesizes an answer grounded in the retrieved records rather than relying solely on its training data.

Because we are working with **raw, unresolved records**, the same real-world entity may appear multiple times with slightly different names, dates, or addresses.  The LLM must reconcile these inconsistencies on the fly, which can lead to incomplete or contradictory answers — a limitation we will address in later notebooks by applying entity resolution before building the RAG.

In [ ]:
SYSTEM_PROMPT = (
    "Answer questions about a corporate ownership and sanctions dataset. "
    "You have access to individual raw source records from Open Sanctions and "
    "Open Ownership. These records have NOT been entity-resolved — the same "
    "real-world entity may appear as separate records with slightly different "
    "names or details. You do not have relationship or graph data beyond what "
    "is mentioned in individual records."
)


def ask_raw_rag(question: str, provider: str = "anthropic", top_k: int = 10) -> None:
    """
    RAG over raw (non-entity-resolved) records.

    Parameters
    ----------
    question : str
        The user's question.
    provider : str
        LLM backend — "anthropic" (Claude) or "openai" (GPT).
    top_k : int
        Number of records to retrieve.
    """
    if provider not in ("anthropic", "openai"):
        raise ValueError(f"provider must be 'anthropic' or 'openai', got '{provider}'")

    print(f"\nQuestion: {question}")
    print(f"Provider: {provider}")
    print("=" * 70)

    # Step 1: Vector search
    question_embedding = embedding_model.encode(question).tolist()
    results = table.search(question_embedding).limit(top_k).to_list()
    print(f"Retrieved {len(results)} records")

    # Step 2: Build context
    context_parts = ["RAW SOURCE RECORDS:\n"]
    for r in results:
        context_parts.append(f"- [{r['data_source']}] {r['text']}")
        context_parts.append("")
    context = "\n".join(context_parts)

    # Step 3: Query LLM
    print("Querying LLM...")
    user_message = f"Context:\n{context}\n\nQuestion: {question}"

    if provider == "anthropic":
        response = anthropic_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=2048,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_message}],
        )
        answer = response.content[0].text
    else:
        response = openai_client.chat.completions.create(
            model="gpt-5.4-nano",
            max_tokens=2048,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_message},
            ],
        )
        answer = response.choices[0].message.content

    print("\n" + "=" * 70)
    print("ANSWER")
    print("=" * 70)
    print(answer)
    print("=" * 70)

## Interactive Query Session

Now we are going to interact with this data via a QA bot.  Be sure to set your `provider` below to `"openai"` to use GPT instead of Claude.  When you are done asking questions, you can type "quit," "exit," or "q" to end.

In [ ]:
provider = "anthropic"  # change to "openai" to use GPT-5.4 nano

print("RAG on Raw Data (No Entity Resolution) - Interactive Session")
print("=" * 70)
print("Ask questions about the corporate ownership and sanctions data.")
print("These are RAW records — no entity resolution has been applied.")
print(f"Current LLM provider: {provider}")
print("Type 'quit' to exit.\n")

while True:
    question = input("Your question: ").strip()

    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break

    if not question:
        continue

    try:
        ask_raw_rag(question, provider=provider)
        print()
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()